In [36]:
import pandas as pd

df = pd.read_csv("data/all_rows_merged.csv")
df

,Probe,Lab_Number,Prediction,Overall_Assessment_pred_file,Overall_Interpretation,Overall_Assessment_data
0,1,P1300010,0,1,Poziom metalicznych produktów zużycia w normie...,WSKAZÓWKA
1,1,P1304884,0,0,Poziom metalicznych produktów zużycia w normie...,W NORMIE
2,1,P1400003,2,2,W efekcie zużycia i/lub korozji poziom miedzi ...,UWAGA
3,1,P1400044,2,2,"P1400044 - próbka oleju z filtra, P1400003 - p...",UWAGA
4,1,P1400113,0,0,NaN,W NORMIE
...,...,...,...,...,...,...
18843,test,P2502080,1,1,Lepkość poniżej zakresu typowego dla klasy SAE...,WSKAZÓWKA
18844,test,P2502129,2,2,Oznaczona lepkość na niskim poziomie. Widmo po...,UWAGA
18845,test,P2505955,0,0,Oznaczona lepkość dla klasy SAE 40 w górnym za...,W NORMIE
18846,test,P2505960,0,0,Oznaczona lepkość dla klasy SAE 40 w normie. T...,W NORMIE


In [37]:
# Wywołanie parsera i zapisanie pliku JSON

from parser import parser
import json

data = parser.df_to_json(df, 0, len(df))

with open("data/parsed_samples.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

In [38]:
from parser.patterns import FEATURE_TO_PATTERN, STATUS_TO_ABNORMALITY

quantified_dict, quantified_df = parser.parser_quantifier(
    parser_output="data/parsed_samples.json",
    feature_to_pattern=FEATURE_TO_PATTERN,
    status_to_abnormality=STATUS_TO_ABNORMALITY,
    return_df=True,
    fill_missing=0,   # 0 oznacza że brak wzmianki to norma
    keep_none=True 
)

In [39]:
quantified_df.to_csv("data/parsed_samples_all_quantified.csv", encoding="utf-8")

In [40]:
import rules

false_negatives = rules.get_false_negatives(data, df, min_negative_params=2)

false_positives = rules.get_false_positives(data, df)

potential_guidelines = rules.get_potential_guidelines(
    data, df,
    suspicious_classes=(0, 2),
    min_borderline_params=1,
    min_total_params=3,
    ok_to_borderline_ratio=1.0,
    require_softening_phrase=False,
)

print("false_positives: ", len(false_positives), false_positives)
print("false_negatives_3: ", len(false_negatives), false_negatives)
print("potential_guidelines: ", len(potential_guidelines), potential_guidelines)

false_positives:  653 ['P1300010', 'P1401076', 'P1401180', 'P1402230', 'P1501440', 'P1502422', 'P1502980', 'P1503769', 'P1600024', 'P1601705', 'P1602482', 'P1602839', 'P1603207', 'P1603214', 'P1603284', 'P1603286', 'P1603294', 'P1603477', 'P1603489', 'P1603491', 'P1603526', 'P1604035', 'P1604043', 'P1604091', 'P1604119', 'P1604235', 'P1604287', 'P1604393', 'P1604495', 'P1604533', 'P1604583', 'P1604677', 'P1604968', 'P1605410', 'P1605427', 'P1605620', 'P1605631', 'P1605632', 'P1605638', 'P1605642', 'P1700327', 'P1700357', 'P1700591', 'P1700691', 'P1700872', 'P1700874', 'P1701267', 'P1701397', 'P1701411', 'P1701436', 'P1701437', 'P1701637', 'P1701926', 'P1702107', 'P1702571', 'P1702579', 'P1702773', 'P1703054', 'P1703620', 'P1704085', 'P1704300', 'P1704347', 'P1704374', 'P1704375', 'P1705390', 'P1705922', 'P1705923', 'P1800048', 'P1800196', 'P1800375', 'P1800418', 'P1800422', 'P1800645', 'P1800646', 'P1800647', 'P1800780', 'P1801492', 'P1801986', 'P1802216', 'P1802574', 'P1803173', 'P180

In [41]:
from utils import labs_to_review_json, labs_to_review_csv

review = labs_to_review_json(false_positives, df, data)

filtered_review = {
    lab: {k: v for k, v in vals.items() if k != "prediction"}
    for lab, vals in review.items()
}


In [42]:

# create dataframe
review_df = labs_to_review_csv(false_positives, df, data)
review_df.to_csv("samples_to_exclude_by_reason/false_positives.csv", index=False, encoding="utf-8")

review_df = labs_to_review_csv(false_negatives, df, data)
review_df.to_csv("samples_to_exclude_by_reason/false_negatives.csv", index=False, encoding="utf-8")

review_df = labs_to_review_csv(potential_guidelines, df, data)
review_df.to_csv("samples_to_exclude_by_reason/potential_guidelines.csv", index=False, encoding="utf-8")